# Understanding JSON


![JSON Logo](images/JSON_vector_logo.svg.png)

&nbsp;

JSON (JavaScript Object Notation) is a lightweight data-interchange format that is easy for humans to read and write, and easy for machines to parse and generate. It is commonly used for transmitting data in web applications, APIs, and configuration files.

Originally derived from JavaScript, JSON has become a language-independent format that is widely supported across programming languages. It represents data as key-value pairs, arrays, and nested structures, making it versatile for various applications.


:::{seealso} What about XML?

XML (eXtensible Markup Language) is another popular data format that was widely used before JSON became prevalent. It uses a more verbose syntax with tags to define elements and attributes. While XML is still used in certain contexts, JSON has largely supplanted it in web development due to its simplicity and efficiency.

This is what a makeup foundation product might look like in XML:

```xml
<product>
    <name>Studio Fix Fluid Foundation</name>
    <brand>MAC Cosmetics</brand>
    <price>49.99</price>
</product>
```

In contrast, the same product in JSON would look like this:

```json
{
  "name": "Studio Fix Fluid Foundation",
  "brand": "MAC Cosmetics",
  "price": 49.99
}
```

As data size grows, JSON's more compact syntax can be more efficient to transmit and parse compared to XML, which is one reason for its widespread adoption in modern web applications.

:::


In [1]:
import requests
import pandas as pd
import numpy as np

## Fetching JSON Data from a URL


We will fetch a JSON file from GitHub and parse it to extract information about makeup foundation products.


In [ ]:
url = "https://raw.githubusercontent.com/bdi593/datasets/main/amazon-makeup-foundation-products/search-results/foundation-page-01.json"

# Send GET request
response = requests.get(url)

# Check if request was successful
if response.status_code == 200:
    data = response.json()  # Load JSON into Python dictionary
    print("Data successfully loaded!")
else:
    raise Exception(f"Failed to fetch data: {response.status_code}")

Data successfully loaded!
Top-level keys:
dict_keys(['requestMetadata', 'productResults', 'pagination'])


:::{warning} Be wary of rate limits!

When fetching data from GitHub, be mindful of their rate limits. If you make too many requests in a short period, you may be temporarily blocked from accessing the data. To follow through the exercise, consider [downloading the dataset from GitHub](https://github.com/bdi593/datasets/tree/main/amazon-makeup-foundation-products) and loading it from your local machine instead of fetching it directly from the URL.

:::


`response.json()` method parses the JSON content of the response and returns it as a Python dictionary (or list, depending on the structure of the JSON). This allows us to easily access and manipulate the data in Python.


In [5]:
type(data)

dict

## General Workflow in Working with JSON Data

You may be working with JSON data that has a strict schema, or you may be working with data that has a more flexible structure. How you approach parsing the data will depend on the structure of the JSON and your specific goals. Here is a general workflow for working with JSON data:


```{mermaid}
flowchart TD
    %% Start Node
    Start([Start: Receive JSON Data]) --> SchemaKnown{Is the JSON<br/>Schema Known?}

    %% --- Known Schema Branch ---
    SchemaKnown -- "Yes" --> K1[Define Data Models<br/>based on Schema]
    K1 --> K2[Select Standard JSON Parser]
    K2 --> K3[Deserialize JSON directly<br/>into Strongly-Typed Objects]
    K3 --> K4[Access data via Object Properties]
    K4 --> End1([Process Data Safely<br/>with Type Safety])

    %% --- Unknown Schema Branch ---
    SchemaKnown -- "No" --> U1[Parse to Generic Structure<br/>e.g., Dictionary]
    U1 --> U2[Inspect Keys and Value Types<br/>at Runtime]
    U2 --> U3{Can a pattern<br/>be inferred?}

    U3 -- "Yes" --> U4[Build Semi-structured<br/>or Flexible Models]
    U3 -- "No" --> U5[Use Dynamic Key Lookups<br/>e.g., data.user.name]

    U4 --> U6[Implement Strict Error Handling<br/>Missing keys, Nulls, Type checks]
    U5 --> U6

    U6 --> End2([Process Data Cautiously])

    %% --- Styling (v9.4.3 compatible) ---
    classDef default fill:#f4f4f9,stroke:#333,stroke-width:1px,color:#333;
    classDef startEnd fill:#d4edda,stroke:#28a745,stroke-width:2px,color:#155724;
    classDef decision fill:#cce5ff,stroke:#007bff,stroke-width:2px,color:#004085;
    classDef knownPath fill:#e2e3e5,stroke:#383d41,stroke-width:1.5px;
    classDef unknownPath fill:#fff3cd,stroke:#ffc107,stroke-width:1.5px;

    class Start,End1,End2 startEnd;
    class SchemaKnown,U3 decision;
    class K1,K2,K3,K4 knownPath;
    class U1,U2,U4,U5,U6 unknownPath;
```


In our case, we don't have a strict schema for the makeup foundation products, so we will be working with a more flexible structure. We will parse the JSON into a generic dictionary and inspect the keys and value types at runtime to understand the structure of the data. This will allow us to extract the information we need while being cautious about potential issues such as missing keys or unexpected data types.


In [4]:
# Inspect the top-level keys
print("Top-level keys:")
print(data.keys())

Top-level keys:
dict_keys(['requestMetadata', 'productResults', 'pagination'])


In [3]:
first_product = data["productResults"][0]

first_product

{'position': 1,
 'asin': 'B01H1V7WQU',
 'title': 'LAURA GELLER NEW YORK Award-Winning Baked Balance-n-Brighten Color Correcting Powder Foundation - Light - Buildable Light to Medium Coverage - Demi-Matte Natural Finish',
 'isSponsored': False,
 'price': {'symbol': '$', 'currentPrice': 19, 'beforePrice': 38},
 'specification': ['0.32 Ounce (Pack of 1)'],
 'image': 'https://m.media-amazon.com/images/I/71zh8GABKAL.jpg',
 'reviews': {'totalReviews': 67800, 'rating': 4.2},
 'badges': {'amazonPrime': False,
  'amazonChoice': False,
  'amazonExclusive': False,
  'bestSeller': False},
 'boughtInPastMonth': '20K+',
 'deliveryInfo': {'freeDelivery': 'Sun, Apr 5',
  'fastestDelivery': 'Tomorrow, Apr 1'},
 'url': 'https://www.amazon.com/dp/B01H1V7WQU'}

## Extract Information from JSON Data

JSON mimics the structure of Python dictionaries and lists, so we can use familiar syntax to access the data. Each product in the dataset is represented as a dictionary with keys such as "asin", "title", and "price". We can iterate through the list of products and extract these details to analyze the makeup foundation products.

As a first exercise, try extracting the ASIN (Amazon Standard Identification Number) and the product title of each product from the JSON data. This will help you get comfortable with navigating the structure of the JSON and accessing specific pieces of information.


First, check the data type of `data["productResults"]` to confirm that it is a list.


In [8]:
type(data["productResults"])

list

Each item in the list is a dictionary representing a product.


In [9]:
type(data["productResults"][0])

dict

Check the keys of the first product to see what information is available.


In [12]:
data["productResults"][0].keys()

dict_keys(['position', 'asin', 'title', 'isSponsored', 'price', 'specification', 'image', 'reviews', 'badges', 'boughtInPastMonth', 'deliveryInfo', 'url'])

In [ ]:
for product in data["productResults"]:
    print(f"ASIN: {product['asin']}, Title: {product['title']}")

ASIN: B01H1V7WQU, Name: LAURA GELLER NEW YORK Award-Winning Baked Balance-n-Brighten Color Correcting Powder Foundation - Light - Buildable Light to Medium Coverage - Demi-Matte Natural Finish
ASIN: B008C13146, Name: Maybelline Dream Fresh Skin Hydrating BB cream, 8-in-1 Skin Perfecting Beauty Balm with Broad Spectrum SPF 30, Sheer Tint Coverage, Oil-Free, Light/Medium, 1 Fl Oz
ASIN: B0BD4MMFVM, Name: GAOY 16ml 2 Pcs Glassy Gel Top Coat and Base Coat Set,No Wipe Foundation Combination for UV Light Cure Nail Polish
ASIN: B0G1T61T7X, Name: Prime Prometics Color Changing Foundation for Mature Women – Instantly Adapts to Your Skin Tone – Buildable Light-to-Medium Coverage – 12-Hour Wear – Natural Dewy Finish – Hypoallergenic
ASIN: B0G1VBDXYF, Name: Luxe Color Changing Foundation for Women – Instantly Adapts to Your Skin Tone – Buildable Light-to-Medium Coverage – 12-Hour Wear – Natural Dewy Finish - Medium
ASIN: B0DJ8Z98MY, Name: Blanc Cover Cream Stick V White - Korean Color-Changing Foun

In [13]:
# Collect data into a list of dictionaries
rows = []

for product in data["productResults"]:
    rows.append({"asin": product.get("asin"), "title": product.get("title")})

df = pd.DataFrame(rows)

df.head()

,asin,title
0,B01H1V7WQU,LAURA GELLER NEW YORK Award-Winning Baked Bala...
1,B008C13146,Maybelline Dream Fresh Skin Hydrating BB cream...
2,B0BD4MMFVM,GAOY 16ml 2 Pcs Glassy Gel Top Coat and Base C...
3,B0G1T61T7X,Prime Prometics Color Changing Foundation for ...
4,B0G1VBDXYF,Luxe Color Changing Foundation for Women – Ins...


:::{seealso} What does `product.get()` do?

The `get()` method is a built-in function for dictionaries in Python. It allows you to retrieve the value associated with a specified key. The advantage of using `get()` over direct key access (e.g., `product["key"]`) is that it does not raise a `KeyError` if the key is not found. Instead, it returns `None` or a specified default value.

:::
